# Problema Inverso Transiente

Este *notebook* tem como objetivo implementar uma PINN para resolver um problema inverso transisente, ou seja, que evolui no tempo.

**Autor**: Edélio Gabriel Magalhães de Jesus.

---

> ATENÇÃO: Esse *notebbok* será melhor aproveitado de for lido após o *notebbok* `04_inverse_stationary.ipynb`, onde foi explicado o que é um problema inverso e implementada uma PINN a um problema inverso estacionário 1D.
>
> Aqui, a diferença será apenas a inclusão de uma condição inicial, o que implica na evolução temporal do problema.

---

## O nosso problema

Como discutido anteriormente, problemas inversos buscam determinar causas desconhecidas a partir de efeitos observados. Nesse exemplo, trabalharemos com a **equação de difusão 2D transiente** — descrevendo o espalhamento de uma espécie química em um meio homogêneo. Ele foi inspirado na implementação discutida em [[ref]](#artigo-base). No artugo original, os autores preveem o parâmetro D enquanto uma função, aqui, simplificamos para D constante, o que significa um meio homogêneo.

---

### `Requisitos teóricos`

#### **Contexto físico**

Quando uma concentração localizada de uma espécie química é introduzida em um meio — por exemplo, uma proteína em solução aquosa — ela se espalha espontaneamente ao longo do tempo devido ao movimento térmico das moléculas. Esse fenômeno é chamado de **difusão** e é governado pela segunda lei de Fick.

O coeficiente de difusão $D$ quantifica a velocidade desse espalhamento:

- materiais com $D$ alto se difundem rapidamente;
- materiais com $D$ baixo se difundem lentamente.

Conhecer $D$ é fundamental em diversas aplicações:

- transporte de fármacos em tecidos biológicos;
- caracterização de biomoléculas em solução;
- processos de separação e filtração.

<div style="text-align: center;">
  <img 
    src="https://upload.wikimedia.org/wikipedia/commons/4/4d/DiffusionMicroMacro.gif"
    alt="Representação Molecular de difusão"
    style="max-width: 400px; width: 50%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://pt.wikipedia.org/wiki/Difus%C3%A3o_molecular' target='_blank'>
    Wikipedia — Difusão Molecular
  </a>
</div>

#### **A equação de difusão 2D**

A evolução temporal do campo de concentração $c(x, y, t)$ é descrita pela equação de difusão:

$$
\frac{\partial c}{\partial t} = D\left(\frac{\partial^2 c}{\partial x^2} + \frac{\partial^2 c}{\partial y^2}\right), \quad (x,y) \in [0,1]^2, \quad t \in [0,1] \tag{X}
$$

onde $D$ é o coeficiente de difusão — assumido constante e homogêneo neste exemplo.

A condição inicial é um **pacote gaussiano** centrado em $(0.5, 0.5)$:

$$
c(x, y, 0) = \exp\left(-\frac{(x-0.5)^2 + (y-0.5)^2}{2\sigma^2}\right) \tag{X}
$$

representando uma concentração localizada no centro do domínio no instante inicial. As condições de contorno são de Dirichlet homogêneas:

$$
c = 0 \quad \text{em toda a fronteira } \partial\Omega \tag{X}
$$

À medida que o tempo avança, o pacote gaussiano se alarga e sua amplitude diminui — a concentração se redistribui uniformemente até se dissipar nas bordas.

#### **O problema inverso**

Em um experimento real, medir $D$ diretamente não é trivial. O que se observa são perfis de concentração $c(x, y, t)$ em alguns instantes — por exemplo, via microscopia de fluorescência ou imagens de concentração.

O problema inverso que propomos é: **a partir de medições esparsas e ruidosas de $c(x, y, t)$ no interior do domínio espaço-temporal, recuperar $D$**.

Matematicamente:

- problema direto:

$$D \longrightarrow c(x, y, t)$$

- problema inverso:

$$c(x, y, t) \longrightarrow D$$

Esse cenário aparece diretamente em aplicações como:

- caracterização de coeficientes de difusão de proteínas em solução [[ref]](#thakur-paper);
- transporte de fármacos em tecidos heterogêneos;
- análise de processos de mistura em microfluídica.

Nesse exemplo, utilizaremos medições sintéticas ruidosas geradas a partir da solução numérica para estimar o valor desconhecido de $D$.

> ⚠️ **Diferença em relação ao exemplo anterior:** no problema de Poisson-Boltzmann, o parâmetro desconhecido era uma **condição de contorno** ($\tilde{\psi}_0$). Aqui, $D$ aparece diretamente na **EDP** — o que torna o gradiente que chega até $D$ durante o treinamento mais indireto, passando pelas derivadas espaciais da rede.

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex05_pinn_inverse_transient.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [12]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### **Importações necessárias**

In [13]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_transient_2d
from ex05_pinn_inverse_transient import (
    numerical_solution_diffusion_2d,
    generate_synthetic_data_diffusion,
    pde_residual_diffusion_2d,
    loss_fn_diffusion,
    train_diffusion,
    evaluate_diffusion
)
from plot_utils import plot_loss, plot_D_evolution, plot_diffusion_snapshots


### **Parâmetros do problema**

Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados na discussão presente artigo original de "Raissi et. al. ("**Data-driven solutions of nonlinear partial differential equations**"[[ref]](#original-paper)), apenas para ter uma base.

As condições de contorno são:

$$
\tilde{\psi}(0) = \tilde{\psi}_0
$$

$$
\tilde{\psi}(\infty) = 0
$$

Além disso, em implementações numéricas, não é possível trabalhar com um domínio infinito. Assim, truncamos o domínio em um valor suficientemente grande:

$$
\tilde{x} \in [0,L]
$$

onde escolhemos:

$$
L \approx 5
$$

pois após alguns comprimentos de Debye o potencial já é praticamente nulo.

In [14]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS   = 3      
N_OUTPUTS  = 1
N_HIDDEN   = 32
N_LAYERS   = 4
ACTIVATION = nn.Tanh

# Parâmetros do problema
D_TRUE   = 0.01
SIGMA    = 0.1

# Parâmetros de amostragem
N_COLLOC = 5000
N_IC     = 500
N_BC     = 200
N_OBS    = 200
NOISE    = 0.01

# Parâmetros para o treinamento
N_EPOCHS = 10000
LR       = 1e-3
W_PDE      = 1.0
W_BC       = 1.0
W_DATA     = 1.0

Usando: cpu


### **Instanciando o modelo**

In [15]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION).to(DEVICE)
D     = nn.Parameter(torch.tensor([0.5], dtype=torch.float32, device=DEVICE))

> ### OBSERVE QUE...

...aqui surge a principal diferença entre uma PINN para um problema direto e uma PINN para um problema inverso.

No problema direto, todos os parâmetros físicos da equação são conhecidos, e a rede neural aprende apenas a função solução $\psi(x)$.

Já no problema inverso, parte da física é desconhecida. Nesse caso, além dos pesos e vieses da rede neural, também queremos estimar um parâmetro físico do sistema — aqui, o potencial de superfície $\psi_0$.

Isso é feito através da instrução:

```python
psi0 = nn.Parameter(torch.tensor([1.0], dtype=torch.float32, device=DEVICE))

### **Amostragem dos pontos**

In [16]:
# ── Solução numérica ───────────────────────────────────────────────────────────
x_ref, y_ref, t_ref, C_ref = numerical_solution_diffusion_2d(D=D_TRUE)

# ── Amostragem ─────────────────────────────────────────────────────────────────
IC_FN = lambda x, y: torch.exp(
    -((x - 0.5)**2 + (y - 0.5)**2) / (2 * SIGMA**2)
)

X_COLLOC              = sample_collocation_rectangular(N_COLLOC, [0,0,0], [1,1,1], DEVICE)
X_IC, C_IC, X_BC, C_BC = sample_boundary_rectangular_transient_2d(
    N_IC, N_BC, 0, 1, 0, 1, 0, 1, IC_FN, DEVICE
)
X_OBS, C_OBS = generate_synthetic_data_diffusion(
    C_ref, x_ref, y_ref, t_ref, N_OBS, NOISE, DEVICE
)

### **Instanciando o otimizador**

In [17]:
optimizer = torch.optim.Adam(
    list(model.parameters()) + [D],
    lr=LR
)

### **Treinamento**

In [18]:
history = train_diffusion(
    model, D, optimizer,
    X_COLLOC, X_IC, C_IC, X_BC, C_BC, X_OBS, C_OBS,
    N_EPOCHS
)

Epoch 00000 | Loss: 2.80e-01 | Loss PDE: 1.97e-03 | Loss IC: 1.27e-01 | Loss BC: 5.21e-02 | Loss data: 9.86e-02 | D: 0.4990
Epoch 00100 | Loss: 4.66e-02 | Loss PDE: 1.48e-04 | Loss IC: 2.99e-02 | Loss BC: 1.32e-03 | Loss data: 1.52e-02 | D: 0.4870
Epoch 00200 | Loss: 4.55e-02 | Loss PDE: 6.08e-05 | Loss IC: 2.95e-02 | Loss BC: 1.11e-03 | Loss data: 1.48e-02 | D: 0.5108
Epoch 00300 | Loss: 4.48e-02 | Loss PDE: 1.31e-04 | Loss IC: 2.92e-02 | Loss BC: 1.08e-03 | Loss data: 1.44e-02 | D: 0.4747
Epoch 00400 | Loss: 4.40e-02 | Loss PDE: 1.63e-04 | Loss IC: 2.88e-02 | Loss BC: 1.01e-03 | Loss data: 1.41e-02 | D: 0.3637
Epoch 00500 | Loss: 4.15e-02 | Loss PDE: 2.60e-04 | Loss IC: 2.74e-02 | Loss BC: 7.89e-04 | Loss data: 1.31e-02 | D: 0.1952
Epoch 00600 | Loss: 3.43e-02 | Loss PDE: 3.86e-04 | Loss IC: 2.22e-02 | Loss BC: 1.44e-03 | Loss data: 1.03e-02 | D: 0.0390
Epoch 00700 | Loss: 3.10e-02 | Loss PDE: 3.75e-04 | Loss IC: 2.01e-02 | Loss BC: 1.13e-03 | Loss data: 9.38e-03 | D: 0.0273
Epoch 00

In [21]:
# após o treinamento
X_test = sample_collocation_rectangular(100, [0,0,0], [1,1,1], DEVICE)
residual = pde_residual_diffusion_2d(model, D, X_test)
print("Resíduo médio:", residual.mean().item())
print("c_xx + c_yy médio:", )

# separe as derivadas
c = model(X_test)
grads = torch.autograd.grad(c, X_test, 
    grad_outputs=torch.ones_like(c), create_graph=True)[0]
c_x = grads[:, 0].unsqueeze(1)
c_y = grads[:, 1].unsqueeze(1)
c_xx = torch.autograd.grad(c_x, X_test,
    grad_outputs=torch.ones_like(c_x), create_graph=True)[0][:, 0]
c_yy = torch.autograd.grad(c_y, X_test,
    grad_outputs=torch.ones_like(c_y), create_graph=True)[0][:, 1]
print("c_xx + c_yy médio:", (c_xx + c_yy).mean().item())
print("c_t médio:", grads[:, 2].mean().item())

Resíduo médio: 0.00104851636569947
c_xx + c_yy médio:
c_xx + c_yy médio: -23.658729553222656
c_t médio: 0.00013574284093920141


### **Visualizando os resultados do treinamento**

In [19]:
results = evaluate_diffusion(model, D, D_TRUE, C_ref, x_ref, y_ref, t_ref, DEVICE)

print(f"D verdadeiro:    {D_TRUE:.4f}")
print(f"D recuperado:    {results['D_pred']:.4f}")
print(f"Erro percentual: {results['error_pct']:.2f}%")
print(f"Erro L2:         {results['l2_error']:.2e}")

# ── Plots ──────────────────────────────────────────────────────────────────────
plot_loss(history)
plot_D_evolution(history, D_TRUE)
plot_diffusion_snapshots(results)

D verdadeiro:    0.1000
D recuperado:    0.0000
Erro percentual: 99.96%
Erro L2:         1.23e+00


Observe que, mesmo com muitas flutuações nas perdas, em média, todas convergiram para valores muito bons, com a perda total ficando na ordem de grandeza de $10^{-3}.

---

Vamos validar nosso modelo a partir da solução analítica.

### **Validando o modelo**

Vamos primeiro gerar a solução numérica para o nosso problema.

## Referências

<a id='artigo-arxiv-pinn'></a> WANG, Tao et al. Physics-Informed Neural Networks for Solving Forward and Inverse Problems Governed by Partial Differential Equations. arXiv preprint arXiv:2403.03970, 2024. Disponível em: https://arxiv.org/html/2403.03970v1